<a href="https://colab.research.google.com/github/jshogland/SpatialModelingTutorials/blob/main/Notebooks/EVALIDator_TreeMap_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparison between TreeMap and EVALIDator summaries
### In this notebook we will compare and contrast estimates derived from EVALIDator and TreeMap at various spatial scales.
##### John Hogland 10/18/2026
Import packages

In [ ]:
import geopandas as gpd, ee, geemap

### Specify latitude and longitude of the Malheur sawmill location and buffer 100 miles around the sawmill
Display the sawmill location and haul radius

In [ ]:
# db_path='./FIAvsEvalidator.gpkg'
# lynm='Malheur Sawmill'

lat, lon= [44.421,-118.994]
sm=gpd.GeoSeries.from_xy([lon],[lat],crs='EPSG:4326')
buf=sm.to_crs('EPSG:5070').buffer(160934.4).to_crs('EPSG:4326')
m=buf.explore( style_kwds={'color':'blue','fillColor': 'none'},tiles='CyclOSM')
m=sm.explore(color='orange', m=m, style_kwds={'weight':10})
m

## To get summarized FIA and TreeMap values we will use FIA's EVALIDator API and Google Earth Engine.
### Let's start with summarizing net cubic feet for all forested lands within 100 miles of the Malheur sawmill  

In [ ]:
import requests
import pandas as pd


# this function will accept a FIADB-API fullreport URL and return dataframes for the estimates as well as subtotals, and totals where available.
def fiadb_api_POST(parameterDictionary):
    # make request
    resp = requests.post(r"https://apps.fs.usda.gov/fiadb-api/fullreport",data=parameterDictionary)
    # parse response to json
    data = resp.json()

    # create output dictionary and populate it with estimate data frames
    outDict = {}
    # append estimates
    outDict['estimates'] = pd.DataFrame(data['estimates'])

    # append subtotals and totals if present
    if 'subtotals' in data.keys():
        subT = {}
        for i in data['subtotals'].keys():
            subT[i] = pd.DataFrame(data['subtotals'][i])
        outDict['subtotals'] = subT
        outDict['totals'] = pd.DataFrame(data['totals'])

    # append metadata
    outDict['metadata'] = data['metadata']
    return outDict


# example of usage 'lat':44.421, 'lon':-12-118.994, 'radius':100,
#requestParameters = { 'reptype':'Circle','lat':44.421, 'lon':-118.994, 'wc':"412023",'radius':100,'snum':16, 'sdenom':2, 'rselected':'Distance to road', 'cselected':'Ownership group', 'outputFormat':'NJSON'}
requestParameters = { 'reptype':'Circle','lat':44.421, 'lon':-118.994, 'wc':"412023",'radius':100,'snum':16, 'sdenom':2, 'outputFormat':'NJSON'}
getData = fiadb_api_POST(parameterDictionary=requestParameters)

# estimate data frame
dfest=getData['estimates']
print('Cubic Foot Volume Esitimates from EVALIDator API for Malheur Sawmill 100 mile buffer')
dfest[['DENOMINATOR_ESTIMATE','NUMERATOR_ESTIMATE','NUMERATOR_SE','RATIO_ESTIMATE','RATIO_SE','NUMERATOR_PLOT_COUNT']]
#dfest.pivot(index='GRP1', columns='GRP2', values=['RATIO_ESTIMATE','RATIO_SE'])

# # list of subtotal data frames
# df_sub_est=getData['subtotals']
# df_sub_est['GRP1']

# # totals data frame
# getData['totals']

# # list all estimate metadata elements
# getData['metadata']

# # access metadata elements
# getData['metadata']['sql']

## Now let's look at what we get using TreeMap
#### Initialize Google Earth Engine

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-jshogland') #you will want to select your personal cloud project

### Specify the TreeMap Data, year, and layer

In [ ]:
url="projects/gtac-data-publish/assets/TreeMap/Product_Version/2026-1"
bnd='VOLCFNET_L'
img_col=ee.ImageCollection(url)
image=img_col.select(bnd).filterDate('2023-01-01', '2023-12-31').first()
mn=image.reduceRegion(reducer=ee.Reducer.mean(), geometry=buf.geometry[0].__geo_interface__,scale=30, maxPixels=1e13)
tn=image.reduceRegion(reducer=ee.Reducer.sum(), geometry=buf.geometry[0].__geo_interface__,scale=30, maxPixels=1e13)
cls=image.reduceRegion(reducer=ee.Reducer.count(), geometry=buf.geometry[0].__geo_interface__,scale=30, maxPixels=1e13)
se=image.reduceRegion(reducer=ee.Reducer.stdDev(), geometry=buf.geometry[0].__geo_interface__,scale=30, maxPixels=1e13)
print ('Mean Cubic Foot Volume from TreeMap for Malheur Sawmill 100 mile buffer: ',mn.getInfo())
print ('Total Cubic Foot Volume from TreeMap for Malheur Sawmill 100 mile buffer: ',tn.getInfo())
print ('Standard Error from TreeMap for Malheur Sawmill 100 mile buffer: ',se.getInfo())
print ('Total Cells: ',cls.getInfo())
m2=geemap.Map(center=[44.421,-118.994], zoom=8,basemap='CyclOSM')
m2.addLayer(geemap.gdf_to_ee(gpd.GeoDataFrame(geometry=buf,crs='EPSG:4326')), {'color':'blue','opacity':0.3}, '100 mile buffer')
m2.addLayer(image, {'min':0, 'max':100, 'palette':['white','green']}, bnd)
m2.addLayer(geemap.gdf_to_ee(gpd.GeoDataFrame(geometry=sm,crs='EPSG:4326')), {'color':'yellow'}, 'Sawmill')
m2

## Include roads distance into EVALIDator analysis

In [ ]:
requestParameters = { 'reptype':'Circle','lat':44.421, 'lon':-118.994, 'wc':"412023",'radius':100,'snum':16, 'sdenom':2, 'rselected':'Distance to road', 'outputFormat':'NJSON'}
getData = fiadb_api_POST(parameterDictionary=requestParameters)

# estimate data frame
dfest=getData['estimates']
print('Cubic Foot Volume Esitimates from EVALIDator API for Malheur Sawmill 100 mile buffer and within 0.5 miles of a road')
dfest_sub=dfest[['GRP1','DENOMINATOR_ESTIMATE','NUMERATOR_ESTIMATE','NUMERATOR_SE','RATIO_ESTIMATE','RATIO_SE','NUMERATOR_PLOT_COUNT','DENOMINATOR_PLOT_COUNT']].iloc[0:5]
dfest_sub

### Estimate weighted averages for group summaries (e.g., sums and averages for only roads within 0.5 miles)

In [ ]:
cf=dfest_sub.NUMERATOR_ESTIMATE.sum()
acres=dfest_sub.DENOMINATOR_ESTIMATE.sum()
plots=dfest_sub.NUMERATOR_PLOT_COUNT.sum()
print('Weighted Average Ratio: ',(dfest_sub.RATIO_ESTIMATE*dfest_sub.DENOMINATOR_PLOT_COUNT).sum()/dfest_sub.DENOMINATOR_PLOT_COUNT.sum())
print('Weighted Standard Error: ',(dfest_sub.RATIO_SE*dfest_sub.DENOMINATOR_PLOT_COUNT).sum()/dfest_sub.DENOMINATOR_PLOT_COUNT.sum())
print('Total Cubic Foot Volume from EVALIDator API for Malheur Sawmill 100 mile buffer and within 0.5 miles of a road: ',cf)
print('Total Acres from EVALIDator API for Malheur Sawmill 100 mile buffer and within 0.5 miles of a road: ',acres)
print('Total Plots from EVALIDator API for Malheur Sawmill 100 mile buffer and within 0.5 miles of a road: ',plots)


## Overlay Tiger roads, Malheur sawmill, and TreeMap Net Volume (CF)

In [ ]:
rds="TIGER/2016/Roads"
rds=ee.FeatureCollection(rds).filterBounds(buf.geometry[0].__geo_interface__)
rds_dist_msk=rds.distance(804.672).clip(buf.geometry[0].__geo_interface__).lte(804.672)
m2=geemap.Map(center=[44.421,-118.994], zoom=8,basemap='CyclOSM')
img_clip=image.updateMask(rds_dist_msk)
m2.addLayer(geemap.gdf_to_ee(gpd.GeoDataFrame(geometry=buf,crs='EPSG:4326')), {'color':'blue','opacity':0.3}, '100 mile buffer')
m2.addLayer(img_clip, {'min':0, 'max':100, 'palette':['white','green']}, bnd)
#m2.addLayer(rds, {'color':'red'}, 'Roads')
m2.addLayer(geemap.gdf_to_ee(gpd.GeoDataFrame(geometry=sm,crs='EPSG:4326')), {'color':'yellow'}, 'Sawmill')
m2

## Spatially summarize TreeMap values

In [ ]:
mn=img_clip.reduceRegion(reducer=ee.Reducer.mean(), maxPixels=1e13,scale=30)
tn=img_clip.reduceRegion(reducer=ee.Reducer.sum(), maxPixels=1e13,scale=30)
se=img_clip.reduceRegion(reducer=ee.Reducer.stdDev(), maxPixels=1e13,scale=30)
cls=img_clip.reduceRegion(reducer=ee.Reducer.count(), maxPixels=1e13,scale=30)

print ('Mean Cubic Foot Volume from TreeMap for Malheur Sawmill 100 mile buffer 0.5 miles from a road: ',mn.getInfo())
print ('Total Cubic Foot Volume from TreeMap for Malheur Sawmill 100 mile buffer: ',tn.getInfo()) #*0.22239484 to adjust for acres per cell
print ('Standard Error from TreeMap for Malheur Sawmill 100 mile buffer: ',se.getInfo())
print ('Total Cells: ',cls.getInfo())
print('Total area of road buffer:', rds_dist_msk.reduceRegion(reducer=ee.Reducer.count(), maxPixels=1e13,scale=30).getInfo())
